# Perfilamento de Carga e Interrupções da Rede Regional com PROC CHART

## Resumo executivo

Um analista de operações de rede usa o PROC CHART para perfilar uma amostra sintética de leituras de medidores de circuitos alimentadores em quatro regiões de atendimento e quatro fontes de geração. O notebook percorre gráficos de barras verticais, barras horizontais, blocos e pizza para resumir a matriz de geração, comparar carga média e total por região, expor o pico de demanda noturno por hora e classificar os minutos de interrupção por fonte — o tipo de exploração rápida com saída em texto que uma equipe de energia e utilidades executa antes de se comprometer com um relatório gráfico. O passo DATA solicita 1,200 linhas; este notebook foi renderizado em modo não licenciado, que limita o conjunto de dados de trabalho às primeiras 100 leituras, portanto todos os gráficos abaixo resumem essa amostra de 100 linhas.

## Fontes de dados

| Conjunto de dados | Linhas | Descrição |
|---------|------|-------------|
| `grid_ops` | 100 (amostra sintética) | Uma linha por leitura de medidor de circuito alimentador em uma rede regional, gerada inline com `call streaminit(20260531)` e `rand()`. O laço do passo DATA solicita 1,200 leituras, mas o modo não licenciado limita o conjunto de dados salvo a 100 observações, portanto os gráficos abaixo descrevem essas 100 linhas. |

# Perfilamento de Carga e Interrupções da Rede Regional com PROC CHART

O PROC CHART produz gráficos de barras, blocos e pizza baseados em caracteres diretamente na listagem — sem necessidade de um dispositivo ODS Graphics. Isso o torna uma ferramenta rápida de exploração inicial para uma equipe de operações de rede que quer *ver* o formato de seus dados de carga e confiabilidade antes de construir visuais refinados com GCHART ou SGPLOT.

Neste notebook nós:

1. Geramos um mês sintético de leituras de medidores de circuitos alimentadores para um operador de rede regional.
2. Traçamos a **matriz de geração** (participação das leituras por fonte).
3. Comparamos a **carga média e total entregue** entre regiões de atendimento.
4. Expomos o **pico de demanda noturno** por hora do dia.
5. Usamos um **gráfico de blocos** para cruzar região com fonte de geração.
6. Classificamos os **minutos de interrupção** por fonte para encontrar a alimentação menos confiável.

Cada instrução e opção abaixo é sintaxe padrão do PROC CHART do SAS 9.4.

## Passo 1 — Gerar os dados sintéticos de operações

O passo DATA abaixo fabrica leituras de medidores em um laço de 1,200 iterações. Cada leitura recebe uma região de atendimento, uma fonte de geração (Gás domina, com Eólica, Solar e Hídrica compondo o restante) e uma hora do dia. A carga é maior na janela noturna das 17:00–21:00 (e sinalizamos essas leituras como pico), e extraímos os minutos de interrupção de uma distribuição de Poisson. `call streaminit` fixa a semente aleatória para que os dados sejam reproduzíveis.

O NOTE no log mostra o resultado prático: esta execução está em modo não licenciado, portanto o conjunto de dados salvo `grid_ops` é limitado às primeiras 100 dessas leituras (100 linhas, 7 colunas). Cada gráfico que segue resume essa amostra de 100 linhas, e cada log do PROC CHART confirma que leu 100 linhas.

In [1]:
/* Operações sintéticas de circuitos alimentadores para um operador de rede regional */
DADOS grid_ops;
    CHAMAR streaminit(20260531);
    COMPRIMENTO region $16 source $12 peak_flag $6;
    VETOR regions[4] $16 _temporary_
        ('Norte','Sul','Leste','Oeste');
    FAZER meter_id = 1 ATÉ 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        SE u < 0.42 ENTÃO source = 'Gás';
        SENÃO SE u < 0.70 ENTÃO source = 'Eólica';
        SENÃO SE u < 0.88 ENTÃO source = 'Solar';
        SENÃO source = 'Hídrica';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'Norte') + 3 * (region = 'Oeste');
        SE hour >= 17 E_LÓGICO hour <= 21 ENTÃO FAZER;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Sim';
        FIM;
        SENÃO FAZER;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'Não';
        FIM;
        SE load_mwh < 0 ENTÃO load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        SAÍDA;
    FIM;
    REMOVER r u BASE;
EXECUTAR;


NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.30 seconds
  cpu   0.30 seconds


## Passo 2 — Matriz de geração

Que participação das leituras cada fonte de geração representa? Um gráfico de barras verticais com `TYPE=PERCENT` responde a isso diretamente: as alturas das barras são a porcentagem de todas as observações que caem em cada categoria de fonte. Como `source` é uma variável de caractere, o PROC CHART trata seus valores como categorias discretas automaticamente.

In [2]:
PROCEDIMENTO chart DADOS=grid_ops;
    VBAR source / type=PERCENT;
    RÓTULO source="Fonte";
EXECUTAR;


Percent of Fonte

         |                    ********          
         |                    ********          
   40.00 +                    ********          
         |                    ********          
         |                    ********          
   30.00 +                    ********          
         |          ********  ********          
         |          ********  ********          
   20.00 +          ********  ********          
         |          ********  ********          
         |********  ********  ********          
         |********  ********  ********  ********
   10.00 +********  ********  ********  ********
         |********  ********  ********  ********
         |********  ********  ********  ********
         |                                      
         +--------------------------------------+
          Hídrica    Eólica     Gás      Solar  
                           Fonte





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 3 — Carga média entregue por região

Agora mudamos de contagem para resumir uma medição. Nomear `load_mwh` em `SUMVAR=` com `TYPE=MEAN` faz com que o comprimento da barra seja a carga média por região, e solicitamos as colunas de estatísticas explicitamente: `MEAN` imprime a média ao lado de cada barra e `CFREQ` adiciona uma coluna de frequência cumulativa. Norte carrega a maior carga média entregue (média em torno de 28.0 MWh), consistente com o deslocamento regional embutido nos dados, enquanto Sul é a menor (cerca de 19.8 MWh).

Também passamos `ASCENDING` para tentar ordenar as barras da menor para a maior média. A reordenação de barras pela estatística do gráfico ainda não é aplicada de forma confiável nesta versão do PROC CHART, portanto a ordem em que as barras aparecem não deve ser usada como classificação — tome a classificação sempre pela coluna `Mean` impressa. Nesta amostra as barras saem na ordem Sul, Leste, Oeste, Norte, com médias 19.8, 21.7, 24.2 e 28.0 MWh: Sul registra a menor carga média e Norte a maior.

In [3]:
PROCEDIMENTO chart DADOS=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
    RÓTULO region="Região" load_mwh="Carga (MWh)";
EXECUTAR;


Mean of Região

 Região                                                  Mean           N     Percent
                                                                                     
    Sul  ****************************                   19.80       19.80       21.00
  Leste  *******************************                21.72       41.52       29.00
  Oeste  **********************************             24.17       65.69       23.00
  Norte  ****************************************       28.03       93.72       27.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 4 — Carga total por região

Aqui `TYPE=SUM` faz com que cada barra seja a carga *total* entregue para a região em vez da média, portanto as barras mais altas marcam as regiões que entregam a maior energia agregada nas leituras amostradas. Norte novamente lidera na carga total entregue.

A instrução também solicita `SUBGROUP=peak_flag`, pedindo ao PROC CHART que divida cada barra conforme a leitura caiu ou não na janela de pico noturno. No SAS isso segmenta cada barra com um glifo distinto por nível de subgrupo e imprime uma legenda. Esta versão ainda não renderiza a segmentação por subgrupo (uma lacuna de capacidade documentada), portanto as barras saem como sequências sólidas de `*` sem detalhamento por segmento — os *totais* das barras estão corretos, mas a divisão pico-vs-fora-de-pico não é mostrada aqui. Para ver quanta carga cai nas horas de pico, use a visão por hora do dia no Passo 5.

In [4]:
PROCEDIMENTO chart DADOS=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
    RÓTULO region="Região" load_mwh="Carga (MWh)" peak_flag="Pico";
EXECUTAR;


Sum of Região

         |                     SSSSS
         |                     SSSSS
         |                     SSSSS
     600 +              SSSSS  SSSSS
         |SSSSS         SSSSS  SSSSS
         |SSSSS         SSSSS  SSSSS
         |SSSSS         SSSSS  SSSSS
     400 +NNNNN  SSSSS  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
     200 +NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |                          
         +--------------------------+
          Oeste   Sul   Leste  Norte
                    Região

Symbol  Pico
------  ----
  N     Não 
  S     Sim 





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 5 — Formato da carga ao longo do dia

`hour` é contínua, portanto fixamos a segmentação nós mesmos com uma lista explícita `MIDPOINTS=` em centros de 4 horas (2, 6, 10, 14, 18, 22). Cada barra mostra a carga média entregue para leituras próximas àquela hora. A barra centrada em 18 deve se destacar — esse é o pico de demanda noturno que embutimos nos dados.

In [5]:
PROCEDIMENTO chart DADOS=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
    RÓTULO hour="Hora" load_mwh="Carga (MWh)";
EXECUTAR;


Mean of Hora

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            Hora





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 6 — Região por fonte de geração (gráfico de blocos)

Um gráfico BLOCK renderiza uma pequena tabela de duas entradas como uma grade de blocos. Com `GROUP=source` e `SUMVAR=load_mwh / TYPE=MEAN`, o gráfico cruza cada região contra a fonte de geração que a atende, com a altura do bloco proporcional à carga média — uma forma compacta de identificar quais combinações de região/fonte carregam a maior carga média.

In [6]:
PROCEDIMENTO chart DADOS=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
    RÓTULO region="Região" load_mwh="Carga (MWh)" source="Fonte";
EXECUTAR;


Block chart of Mean of Região by Fonte

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
  Oeste    Sul    Leste   Norte 
             Região





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 7 — Matriz de geração como gráfico de pizza

A mesma informação de participação por fonte do Passo 2, desenhada como uma pizza. PIE com `TYPE=PERCENT` dimensiona cada fatia por sua porcentagem do total de leituras e imprime uma legenda das porcentagens das fatias ao lado da figura.

In [7]:
PROCEDIMENTO chart DADOS=grid_ops;
    PIE source / type=PERCENT;
    RÓTULO source="Fonte";
EXECUTAR;


Pie chart of Fonte

           Fonte     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
         Hídrica       14.00     14.0%  ####
          Eólica       28.00     28.0%  ********
             Gás       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 8 — Minutos de interrupção por fonte

Por fim, classificamos a confiabilidade. `SUMVAR=outage_min` com `TYPE=SUM` totaliza os minutos de interrupção por fonte. Passamos `DESCENDING` para tentar levar a fonte de pior desempenho para o topo, mas como no Passo 3 as barras não são reordenadas — elas são impressas em ordem de categoria (Hídrica, Eólica, Gás, Solar) e a reordenação de barras ainda não está implementada. Leia a classificação a partir da coluna `Sum` impressa: Gás responde pela maior quantidade de minutos totais de interrupção nesta amostra (122), bem à frente de Eólica (64), Solar (43) e Hídrica (38). Isso acompanha a matriz de geração — Gás atende a maior quantidade de leituras, portanto acumula a maior quantidade de minutos de interrupção no geral.

In [8]:
PROCEDIMENTO chart DADOS=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DECRESCENTE;
    RÓTULO source="Fonte" outage_min="Minutos de interrupção";
EXECUTAR;


Sum of Fonte

   Fonte                                                   Sum        Cum.     Percent
                                                                       Sum            
 Hídrica  ************                                      38          38       14.00
  Eólica  *********************                             64         102       28.00
     Gás  ****************************************         122         224       45.00
   Solar  **************                                    43         267       13.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpretando os resultados

Ler os gráficos em conjunto dá à equipe de operações um quadro situacional rápido (sobre a amostra de 100 linhas que esta execução capturou):

- **Matriz de geração (Passos 2 e 7).** Gás carrega a maior participação das leituras (cerca de 45%), com Eólica em segundo (cerca de 28%) e Hídrica e Solar atrás (cerca de 14% e 13%) — o gráfico de barras verticais e a pizza contam a mesma história de duas formas, uma verificação de sanidade útil.
- **Carga por região (Passos 3 e 4).** O HBAR de carga média mostra Norte com a maior carga média entregue (média ~28 MWh) e Sul a menor (~20 MWh), consistente com o deslocamento regional embutido nos dados. O gráfico SUM confirma que Norte também entrega a maior energia total.
- **Formato da carga diária (Passo 5).** A barra de ponto médio centrada na hora 18 é claramente a mais alta, confirmando o pico de demanda das 17:00–21:00 que embutimos nos dados — exatamente onde uma concessionária focaria a resposta à demanda e o planejamento de capacidade.
- **Confiabilidade (Passo 8).** Totalizar os minutos de interrupção por fonte destaca Gás como o maior contribuinte de tempo de inatividade nesta amostra (122 minutos), o alvo natural seguinte para revisão de manutenção — embora isso reflita principalmente o fato de Gás atender a maior quantidade de leituras.

Duas opções de exibição usadas aqui — reordenação de barras `ASCENDING`/`DESCENDING` (Passos 3 e 8) e segmentação de barras `SUBGROUP=` (Passo 4) — são aceitas pelo analisador mas ainda não renderizadas por esta versão do PROC CHART, portanto as classificações e divisões são lidas a partir das colunas de estatística impressas em vez da ordem ou sombreamento das barras.

O PROC CHART produz saída apenas em caracteres, portanto, para visuais prontos para diretoria, a equipe moveria essas mesmas visões para o PROC GCHART ou PROC SGPLOT. Mas como uma primeira passagem sem configuração sobre uma extração recente, esses gráficos respondem às perguntas operacionais — matriz, magnitude e momento — em segundos.